# Capital Allocation II — Under Uncertainty
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Quantify the sensitivity** of mean-variance weights to estimation error in $\mu$ and $\Sigma$
2. **Apply shrinkage** to expected returns and covariance matrices
3. **Apply Black-Litterman intuition** — mix views with a prior
4. **Apply a robust sizing rule** (the fractional Kelly criterion) under parameter uncertainty
5. **Audit AI-generated optimization outputs** for over-fitting

## 📋 TOC
1. [Setup](#setup)  2. [The Estimation-Error Problem](#problem)
3. [Pitfall Checklist](#pitfalls)  4. [Shrinkage Estimators](#shrinkage)
5. [Fractional Kelly](#kelly)  6. [🎯 Challenge: Robust Sizing](#challenge)
7. [Submission](#submit)  8. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## The Estimation-Error Problem <a id="problem"></a>

In Capital Allocation I we derived $w^* = (1/\gamma) \Sigma^{-1} \mu$.

In Factor Models II we showed that $\mu$ has wide confidence intervals — even
10 years of data gives a Sharpe SE of ~0.1.

**Mean-variance optimization treats $\hat{\mu}$ as the truth.** The
optimal weights swing dramatically with small changes in the input. Real
deployments must use estimators that account for this uncertainty.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Plug-in optimization** | Using $\hat{\mu}$ as if it were $\mu$ produces wildly leveraged weights | Sanity check: do weights exceed +/-2? Then optimization is unreliable |
| 2 | **Inverting a near-singular covariance matrix** | $\Sigma$ with N assets has $N(N+1)/2$ parameters from $T$ observations; needs $T \gg N^2$ for stable inverse | Compute condition number of $\Sigma$; if > 100, inversion is unstable |
| 3 | **No shrinkage** | Sample $\Sigma$ is noisy; shrinkage toward a structured target (identity or factor-based) is almost always better | Compare in-sample vs OOS MVE Sharpe — gap > 2x indicates overfitting |
| 4 | **Ignoring transaction costs in the optimization** | Optimal weights demand 100% turnover; real strategies need penalty | Add a cost penalty proportional to $|\Delta w|$ |

---
## Shrinkage Estimators <a id="shrinkage"></a>

**Mean shrinkage** (James-Stein style): pull $\hat{\mu}$ toward a prior $\mu_0$:

$$\tilde{\mu} = (1-w) \hat{\mu} + w \mu_0$$

Common choice for $\mu_0$: the **grand mean** (average across all assets) or
**zero**. Weight $w$ chosen to minimize MSE.

**Covariance shrinkage** (Ledoit-Wolf): pull $\hat{\Sigma}$ toward a structured
target (identity scaled to average variance, or factor-implied covariance):

$$\tilde{\Sigma} = (1-\alpha) \hat{\Sigma} + \alpha F$$

where $F$ is the target. Optimal $\alpha$ has a closed form (Ledoit-Wolf).

> **💡 Why this helps**
>
> The sample mean and covariance are unbiased but high-variance. Shrinkage
> trades a small bias for a large variance reduction. The resulting
> portfolio weights are dramatically more stable, and OOS performance is
> almost always better.

---
## Fractional Kelly <a id="kelly"></a>

The **Kelly criterion** says bet your wealth such that expected log-wealth
is maximized. For a single asset:

$$f^* = \frac{\mu}{\sigma^2}$$

(This is the same as Merton's optimal weight with $\gamma = 1$.)

**Full Kelly is too aggressive.** Even small estimation error in $\mu$ can
lead to bankruptcy. Real practitioners use **fractional Kelly** — typically
1/2 Kelly or 1/4 Kelly:

$$f^* = \frac{1}{2} \cdot \frac{\hat{\mu}}{\hat{\sigma}^2}$$

Why halve? Because if your $\hat{\mu}$ is biased upward by 50% (very
possible with small samples), full Kelly produces a 4x larger position than
optimal. Fractional Kelly is the **standard robust-sizing rule of thumb**.

In [ ]:
# Demonstrate: sensitivity of weight to noise in mu
np.random.seed(0)
true_mu = 0.05    # true expected excess return per period
true_sig = 0.15
T = 60   # 5 years monthly
gamma = 3

n_sims = 1000
weights_full = []
weights_half = []
for _ in range(n_sims):
    sample = np.random.normal(true_mu / 12, true_sig / np.sqrt(12), T)   # monthly
    mu_hat = sample.mean() * 12
    sig_hat = sample.std() * np.sqrt(12)
    w_full = mu_hat / (gamma * sig_hat**2)
    w_half = 0.5 * w_full
    weights_full.append(w_full)
    weights_half.append(w_half)

print(f"True optimal w = {true_mu / (gamma * true_sig**2):.2f}")
print(f"Full Kelly estimated weights: mean={np.mean(weights_full):.2f}, std={np.std(weights_full):.2f}")
print(f"Half Kelly estimated weights: mean={np.mean(weights_half):.2f}, std={np.std(weights_half):.2f}")
print(f"\n→ Half Kelly cuts variance by 4x, at the cost of bias toward zero.")

---
## 🎯 Challenge: Robust Sizing <a id="challenge"></a>

> **Setup.** You ran a 3-year backtest of your strategy. It produced an
> annualized Sharpe of 1.2. You want to deploy. How much capital?

### Q1 — Naive plug-in size

Apply the plug-in formula $w = \hat{\mu}/(\gamma \hat{\sigma}^2)$ with $\gamma = 3$,
$\hat{\mu} = 0.12$/yr, $\hat{\sigma} = 0.10$/yr.

> **📌 Required:**
> ```python
> mu_hat       = 0.12
> sigma_hat    = 0.10
> gamma        = 3
> w_plugin     = ____
> ```

In [ ]:
mu_hat    = 0.12
sigma_hat = 0.10
gamma     = 3

w_plugin = ____
print(f"Plug-in weight: {w_plugin:.2f}")

### Q2 — Estimate uncertainty

The Sharpe SE on a 3-year sample (T=36 months) with Sharpe=1.2 is:

$$SE = \sqrt{(1 + 1.2^2/2) / 36}$$

> **📌 Required:**
> ```python
> sharpe_se = ____   # apply the formula
> ```

In [ ]:
T = 3 * 12
realized_sharpe = 1.2

sharpe_se = ____
print(f"Sharpe SE: {sharpe_se:.2f}")
print(f"95% CI on Sharpe: [{realized_sharpe - 1.96*sharpe_se:.2f}, {realized_sharpe + 1.96*sharpe_se:.2f}]")

### Q3 — Half-Kelly size

Apply half-Kelly: multiply your plug-in by 0.5.

> **📌 Required:**
> ```python
> w_half_kelly = ____
> ```

In [ ]:
w_half_kelly = ____
print(f"Half-Kelly weight: {w_half_kelly:.2f}")
print(f"vs Plug-in:        {w_plugin:.2f}")

### Q4 — Memo

Max 5 sentences. Recommend a sizing for the new strategy. Cite (i) the plug-in
weight, (ii) the half-Kelly alternative, (iii) the Sharpe uncertainty that
justifies the haircut.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["w_plugin", "sharpe_se", "w_half_kelly", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "CapitalAllocationII_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Mean-variance is fragile.** Optimal weights are very sensitive to $\hat{\mu}$.
2. **Shrinkage helps.** Both for means and covariances. Closed-form formulas exist.
3. **Half-Kelly is the practical sizing rule.** Bias toward zero is cheap; bankruptcy is not.
4. **Backtest Sharpe overstates expected OOS Sharpe.** Discount accordingly.
5. **AI will hand you the "optimal" weights. You decide how much to trust them.**